# Week 1 Day 2 Lab - UnderCover Agent Cost Analysis

A claims handler at UnderCover Travel Insurance has provided data for 25 real travel insurance claims, and has also provided data for the decisions made by human experts for all 25 claims.

Our goal is to create an agent that can consistently mirror these human decisions, with at least 15/25 correct outcomes each time, at an optimum cost. 

To acheive this, we will experiment with different LLMs inside the agent using OpenRouter.

Once we know which LLM helps the agent acheive the desired consistent results, and for the cheapest cost, that agent is the WINNER!

In [ ]:
import json
import os
import time
from pprint import pprint
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI

## Connect to OpenRouter

The OpenRouter key is read from the `.env` file, so please create the `.env` file in your local environment and include your OpenRouter API key before running.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

In [ ]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data():
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.
    """
    claims = load("claims.json")
    decisions = load("evals/decisions.json")
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
claims, labels = get_sample_data()
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

### Explore the data

Always start by exploring and understanding the source data.

In [ ]:
claim = claims[0]
decision = labels[claim["claim_id"]]
print(f"Example claim {claim["claim_id"]}: {claim}")
print(f"Example decision for claim {claim["claim_id"]}: {decision}")

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

These charts show a) claims of each type, split by the handler's decision and b) breakdown by difficulty

In [ ]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

In [ ]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Create our agent with LangChain create_agent

An agent is an LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

### The prompt

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

### The tools

In [ ]:
recorded = {}


@tool # LangChain @tool decorator, turns a function into a tool an LLM can call
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses LangChain's `ChatOpenAI` chat model class, configured to send requests to OpenRouter. This gives us access to models from multiple providers through a single interface. We give the agent a system prompt with instructions and policy wording, plus a line about the tools.

In [ ]:
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

Create an agent creation function that can create our agent. It will use the provided model, and attach the tools and system prompt it needs to give it the capability to process a claim. We make the agent outside of claim processing so that we can create it once and then run the same agent over all claims in a loop, rather than recreating the agent each time. This could be expanded to handle rate limiting etc.

In [ ]:
def make_agent(model):
    """
    Create an agent using the model parameter"""
    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)
    return create_agent(
        model=llm,
        tools=[get_policy, record_decision],
        system_prompt=AGENT_SYSTEM_PROMPT,
        name=model,
    )

Create an invocation function for a single agent run with a specific LLM powered agent and claim. Time it, invoke the agent on the claim, and return the results of the run.

In [ ]:
def decide_with_agent(agent, claim):
    """
    Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()

    error = None
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
        cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    except Exception as e:
        error = str(e)
        cost = 0.0

    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]

    return {
        "model": agent.name,
        "effort": "low",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
        "error": error,
    }

Execute the runner function to create an agent using `luna` only and one claim.

In [ ]:
recorded.clear()

luna_agent = make_agent("openai/gpt-6-luna")
result = decide_with_agent(luna_agent, claims[0])
pprint(result)

Now let's run this on all claims, using `luna`.

In [ ]:
recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(luna_agent, claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")
    

We are interested in the accuracy rate and the total cost. Let's monitor duration too.

In [ ]:
accuracy_count = sum(result["right"] for result in agent_results)
print(f"{accuracy_count} of {len(agent_results)} results are accurate.")

duration = sum(result["seconds"] for result in agent_results)
print(f"Total duration of processing {len(agent_results)} claims is {duration:.2f} seconds.")

total_cost = sum(result["cost"] for result in agent_results)
print(f"Total cost of processing {len(agent_results)} claims is ${total_cost:.4f}.")

So now that we have accuracy and cost of a full run for `luna`, let's try this with different models and compare to see which model holds accuracy at the cheapest cost. `luna` will be hard to beat at under 1 cent for a full run! I have intentionally chosen models with low average costs to give them a chance, especially in these thrifty times. Let's run them all again including `luna` to check that accuracy is consistent.

In [ ]:
MODELS = [
    "openai/gpt-6-luna",
    "anthropic/claude-haiku-5.5",
    "deepseek/deepseek-v4.1-flash",
    "google/gemini-3.5-flash-lite",
    "openrouter/free",
]


In [ ]:
model_results = {}

for model in MODELS:
    recorded.clear()
    agent = make_agent(model) # Create the agent once per model
    model_results[model] = [decide_with_agent(agent, claim) for claim in claims]
    print(f"{len(recorded)} of {len(claims)} claims have a recorded decision using model {model}")

Now lets analyse accuracy and cost for each claim run on each model

In [ ]:
for model, results in model_results.items():
    print(f"Claim processing anaysis for model {model}.")
    accuracy_count = sum(result["right"] for result in results)
    print(f"{accuracy_count} of {len(results)} results are accurate.")

    duration = sum(result["seconds"] for result in results)
    print(f"Total duration of processing {len(results)} claims is {duration:.2f} seconds.")

    total_cost = sum(result["cost"] for result in results)
    print(f"Total cost of processing {len(results)} claims is ${total_cost:.4f}.\n")


### Typical results

openai/gpt-6-luna: 17/25 correct. Duration: 212.70 seconds. Cost: $0.0090.

anthropic/claude-haiku-5.5: 17/25 correct. Duration: 269.20 seconds. Cost: $0.0508.

deepseek/deepseek-v4.1-flash.1: 17/25 correct. Duration: 175.10 seconds. Cost: $0.0342.

google/gemini-3.5-flash-lite: 12/25 correct. Duration: 86.80 seconds. Cost: $0.0935.

openrouter/free: 15/25 correct. Duration: 1152.00 seconds. Cost: $0

So for me the clear winner is Luna. 17/25 accuracy at under a cent with a reasonable duration. Other models with similar accuracy cost 3-5 times as much.  Gemini was too inaccurate and expensive, although fast. Free models had borderline accuracy, but took an age to run.

Other things I would consider if expanding this:
- Handle rate-limiting. Didn't happen often but most likely on openai. Use rate-limiting middleware on agent creation.
- Explore models via openrouter before using. Do they offer tool support, reasoning etc?